In [114]:
import pandas as pd
import numpy as np
# from tqdm import tqdm
import random
import time
import csv
import json
import openai
import os
from tqdm import tqdm
import time
import ast

In [115]:
baseline_results = pd.read_csv("results/baseline_gpt_oss_1000.csv") #change name to baseline_gpt_oss_1000
baseline_results

,Argument ID,values,seconds
0,A01001,"{'Be creative':0, 'Be curious':0, 'Have freedo...",30.924304
1,A01002,"{'Be creative':1, 'Be curious':0, 'Have freedo...",25.552757
2,A01003,"{'Be creative':0, 'Be curious':0, 'Have freedo...",32.587301
3,A01004,"{'Be creative':0, 'Be curious':0, 'Have freedo...",21.698446
4,A01005,"{'Be creative':0, 'Be curious':0, 'Have freedo...",35.828496
...,...,...,...
995,A25476,"{'Be creative':0, 'Be curious':0, 'Have freedo...",37.146281
996,A25480,"{'Be creative':1, 'Be curious':0, 'Have freedo...",23.166566
997,A25486,"{'Be creative':0, 'Be curious':0, 'Have freedo...",24.789036
998,A25493,"{'Be creative':0, 'Be curious':0, 'Have freedo...",35.231369


#### Error Checking


Check if the predictions have been made correctly (dictionary with 54 keys and 54 values that are integers 0 / 1)

In [116]:
# TRANSFORM THE LLMS' OUTPUT INTO A REAL DICTIONARY
# baseline_results

n_errors = 0
for idx, row in baseline_results.iterrows():
    try:
        baseline_results.at[idx, 'values'] = ast.literal_eval(row['values'])
    except SyntaxError:
        pass



In [117]:
expected_keys_in_order = [
    'Be creative', 'Be curious', 'Have freedom of thought', 'Be choosing own goals',
    'Be independent', 'Have freedom of action', 'Have privacy', 'Have an exciting life',
    'Have a varied life', 'Be daring', 'Have pleasure', 'Be ambitious', 'Have success',
    'Be capable', 'Be intellectual', 'Be courageous', 'Have influence',
    'Have the right to command', 'Have wealth', 'Have social recognition',
    'Have a good reputation', 'Have a sense of belonging', 'Have good health',
    'Have no debts', 'Be neat and tidy', 'Have a comfortable life',
    'Have a safe country', 'Have a stable society', 'Be respecting traditions',
    'Be holding religious faith', 'Be compliant', 'Be self-disciplined',
    'Be behaving properly', 'Be polite', 'Be honoring elders', 'Be humble',
    'Have life accepted as is', 'Be helpful', 'Be honest', 'Be forgiving',
    'Have the own family secured', 'Be loving', 'Be responsible',
    'Have loyalty towards friends', 'Have equality', 'Be just',
    'Have a world at peace', 'Be protecting the environment',
    'Have harmony with nature', 'Have a world of beauty', 'Be broadminded',
    'Have the wisdom to accept others', 'Be logical', 'Have an objective view'
]

def is_valid(d):
    return (
        isinstance(d, dict)
        and len(d) == 54 # se han predicho 54 valores
        and list(d.keys()) == expected_keys_in_order # se cumple el orden y los nombres de los valores presentes en la lista original del prompt
        and all(isinstance(v, (int)) for v in d.values()) # es un número entero
        and all(v in (0, 1) for v in d.values()) # los valores son 0 o 1
    )

valid_mask = baseline_results["values"].apply(is_valid)

all_valid = valid_mask.all()

invalid_rows = baseline_results[~valid_mask]

# print all rows from the dataframe that do not follow the specified rules 
print(len(invalid_rows), "arguments have not been predicted according to the specified rules.")

44 arguments have not been predicted according to the specified rules.


In [118]:
invalid_ids = baseline_results.loc[~valid_mask, "Argument ID"].tolist()

print(f"{len(invalid_ids)} arguments have not been predicted according to the specified rules.")
print(invalid_ids)

44 arguments have not been predicted according to the specified rules.
['A01006', 'A01010', 'A01017', 'A02001', 'A03004', 'A05087', 'A07014', 'A07045', 'A07066', 'A07082', 'A07087', 'A08011', 'A09016', 'A09022', 'A09031', 'A09066', 'A12047', 'A12075', 'A12094', 'A12109', 'A12111', 'A06017', 'A07017', 'A12138', 'A12285', 'A12361', 'A18457', 'A18464', 'A18495', 'A19082', 'A19361', 'A20490', 'A21424', 'A21448', 'A21477', 'A22037', 'A22094', 'A22228', 'A22476', 'A23244', 'A24111', 'A24141', 'A24159', 'A25309']


In [119]:
I = 0
for arg, bad_prediction in zip(invalid_rows["Argument ID"],invalid_rows["values"]):
    print("\n\nArg:", arg, "no cumple los requisitos de clasificación:\n")
    # print(bad_prediction)
    try:
        print(len(bad_prediction) == 54)
        print(list(bad_prediction.keys()) == expected_keys_in_order)
        print(all(isinstance(v, (int)) for v in bad_prediction.values())) # es un número entero
        print(all(v in (0, 1) for v in bad_prediction.values())) # los valores son 0 o 1

        dict_keys = set(bad_prediction.keys())
        expected_keys = set(expected_keys_in_order)

        missing = expected_keys - dict_keys
        extra = dict_keys - expected_keys
        print("The error detected is:", extra)
        print()
        print()
    except AttributeError:
        print()
        print(bad_prediction)



Arg: A01006 no cumple los requisitos de clasificación:

True
False
True
True
The error detected is: {'Have an excited life'}




Arg: A01010 no cumple los requisitos de clasificación:

True
False
True
True
The error detected is: {'Have sense of belonging'}




Arg: A01017 no cumple los requisitos de clasificación:

True
False
True
True
The error detected is: {'Have sense of belonging', 'Have safe country', 'Have stable society', 'Have comfortable life'}




Arg: A02001 no cumple los requisitos de clasificación:

True
False
True
True
The error detected is: {'Have a varied life - allowing people to change parts of their life; allowing people to move flat easily; promoting local clubs (sports, ...); providing many activities', 'Have influence - having more people to ask for a favor; resulting in more influence; resulting in more obligations towards the own side; resulting in more ways to control events', 'Be humble - demoting arrogance; demoting to think one deserves more than other peo

### Errors

Your dictionary contains:

'Have an excited life'

But the expected list contains:

'Have an exciting life'


------------------------------

Your dictionary contains:

'Have sense of belonging'

Expected list contains:

'Have a sense of belonging'

--------------------------------

